<a href="https://colab.research.google.com/github/agai-research/trust-mpgnn/blob/main/Trust_MPGNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trust-MPGNN — Conflict-aware Composition of IoT Services
### Metapath-Guided Trust Learning for Intelligent Cyber-Physical Systems
**Reference:** F. Ghedass, H. Mezni, M. Alabdulhafith, H. Elmannai. *Conflict-aware Composition of IoT Services: an Approach based on MetaPath-Guided Trust Learning.* Software: Practice and Experience, Wiley, 2026.

## 1. Setup
Install dependencies and create the project folder structure.

In [1]:
!pip install -q torch networkx numpy
import os
for d in ["src/tkg", "src/gnn", "src/composition", "src/utils",
          "data/raw", "data/instances", "experiments", "tests",
          "output", "logs", "web"]:
    os.makedirs(d, exist_ok=True)
for d in ["src", "src/tkg", "src/gnn", "src/composition", "src/utils"]:
    open(os.path.join(d, "__init__.py"), "a").close()
print("Project structure ready.")

Project structure ready.


## 2. Configuration
All hyperparameters and paths (unchanged from the original `config.json`).

In [2]:
%%writefile config.json
{
  "tkg": {
    "num_providers": 500,
    "num_services": 2000,
    "num_resources": 1000,
    "num_relations": 5000,
    "relation_types": ["TRUST", "SUPPORT", "OPPOSE", "NEUTRAL", "ALLIED", "CONFLICT"]
  },
  "gnn": {
    "embed_dim": 128,
    "num_layers": 2,
    "num_heads": 4,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "weight_decay": 1e-4,
    "batch_size": 256,
    "epochs": 200,
    "trust_threshold": 0.6,
    "sampling_size": 20
  },
  "metapaths": [
    {"name": "PP_trust", "path": [["Provider","TRUST","Provider"]]},
    {"name": "PP_trust2", "path": [["Provider","TRUST","Provider"],["Provider","TRUST","Provider"]]},
    {"name": "SR_support", "path": [["Service","SUPPORT","Resource"]]},
    {"name": "SR_oppose", "path": [["Service","OPPOSE","Resource"]]},
    {"name": "SS_allied", "path": [["Service","ALLIED","Service"]]},
    {"name": "PSR", "path": [["Provider","TRUST","Provider"],["Provider","TRUST","Service"]]},
    {"name": "SRS", "path": [["Service","SUPPORT","Resource"],["Resource","SUPPORT","Service"]]},
    {"name": "PSS", "path": [["Provider","TRUST","Service"],["Service","ALLIED","Service"]]},
    {"name": "PPS", "path": [["Provider","TRUST","Provider"],["Provider","TRUST","Service"]]},
    {"name": "RR_conflict", "path": [["Resource","CONFLICT","Resource"]]}
  ],
  "composition": {
    "max_services": 50,
    "max_resources": 50,
    "qos_weights": {"reliability": 0.4, "response_time": 0.3, "cost": 0.3}
  },
  "experiment": {
    "conflict_densities": [0.1, 0.2, 0.3, 0.4, 0.5],
    "workflow_sizes": [5, 10, 20, 30, 50],
    "icps_sizes": [1000, 1500, 2000, 2500, 3000],
    "trust_thresholds": [0.5, 0.6, 0.7, 0.8]
  },
  "paths": {
    "dataset": "data/raw/dataset.json",
    "tkg_output": "output/tkg.json",
    "embeddings": "output/embeddings.pt",
    "results": "output/results.json",
    "logs": "logs/"
  },
  "author": "H. Mezni"
}


Writing config.json


## 3. Utilities — logging, JSON I/O

In [3]:
%%writefile src/utils/utils.py
"""
utils.py - Utility functions: logging, path resolution, JSON I/O.
Author: H. Mezni
"""

import os
import json
import logging
import time

LOG_FORMAT = "%(asctime)s [%(levelname)s] %(name)s: %(message)s"


def setup_logging(log_dir: str = "logs", level: int = logging.INFO):
    """Configure file + console logging."""
    os.makedirs(log_dir, exist_ok=True)
    ts = time.strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"run_{ts}.log")
    logging.basicConfig(
        level=level,
        format=LOG_FORMAT,
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return log_file


def load_json(path: str) -> dict:
    with open(path) as f:
        return json.load(f)


def save_json(obj, path: str):
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def load_config(path: str = "config.json") -> dict:
    return load_json(path)


def resolve_path(base: str, rel: str) -> str:
    """Resolve a relative path from a base directory."""
    return os.path.normpath(os.path.join(base, rel))


Writing src/utils/utils.py


## 4. Trust Knowledge Graph (`src/tkg/tkg.py`)
*Fix applied: guarded `to_json` against crashing on paths with no directory component.*

In [4]:
%%writefile src/tkg/tkg.py
"""
tkg.py - Trust Knowledge Graph construction using NetworkX.
Loads dataset and builds TKG as a directed heterogeneous graph.
Author: H. Mezni
"""

import json
import os
import logging
import networkx as nx
import numpy as np
from collections import defaultdict

logger = logging.getLogger(__name__)

# Valid relation types as defined in the paper
RELATION_TYPES = ["TRUST", "SUPPORT", "OPPOSE", "NEUTRAL", "ALLIED", "CONFLICT"]


class TrustKnowledgeGraph:
    """
    TKG = G = <V, E, T, phi>
    V = Providers U Services U Resources U Features
    T = {TRUST, SUPPORT, OPPOSE, NEUTRAL, ALLIED, CONFLICT}
    """

    def __init__(self):
        self.G = nx.DiGraph()
        self.providers = {}   # id -> metadata
        self.services = {}
        self.resources = {}
        self.node_index = {}  # id -> int index
        self.index_node = {}  # int -> id
        self.relation_index = {r: i for i, r in enumerate(RELATION_TYPES)}

    def load_from_dataset(self, dataset: dict):
        """Populate TKG from a dataset dict (providers, services, resources, relations)."""
        logger.info("Building Trust Knowledge Graph ...")
        self.providers = {p["id"]: p for p in dataset["providers"]}
        self.services = {s["id"]: s for s in dataset["services"]}
        self.resources = {r["id"]: r for r in dataset["resources"]}

        # Add nodes
        idx = 0
        for p in dataset["providers"]:
            self.G.add_node(p["id"], ntype="Provider", **{k: v for k, v in p.items() if k not in ("feature_vector",)})
            self.node_index[p["id"]] = idx; self.index_node[idx] = p["id"]; idx += 1
        for s in dataset["services"]:
            self.G.add_node(s["id"], ntype="Service", **{k: v for k, v in s.items() if k not in ("feature_vector",)})
            self.node_index[s["id"]] = idx; self.index_node[idx] = s["id"]; idx += 1
        for r in dataset["resources"]:
            self.G.add_node(r["id"], ntype="Resource", **{k: v for k, v in r.items() if k not in ("feature_vector",)})
            self.node_index[r["id"]] = idx; self.index_node[idx] = r["id"]; idx += 1

        # Add edges
        added = 0
        for rel in dataset["relations"]:
            h, t, rtype = rel["head"], rel["tail"], rel["type"]
            if h in self.G and t in self.G and rtype in RELATION_TYPES:
                self.G.add_edge(h, t, relation=rtype, rtype_idx=self.relation_index[rtype])
                added += 1
        logger.info(f"TKG built: {self.G.number_of_nodes()} nodes, {self.G.number_of_edges()} edges.")
        return self

    def load_from_file(self, path: str):
        """Load dataset from JSON file and build TKG."""
        with open(path) as f:
            dataset = json.load(f)
        return self.load_from_dataset(dataset)

    def get_neighbors_by_relation(self, node_id: str, relation: str) -> list:
        """Return neighbors of node_id connected via given relation type."""
        return [v for u, v, data in self.G.out_edges(node_id, data=True)
                if data.get("relation") == relation]

    def get_metapath_neighbors(self, node_id: str, metapath: list) -> list:
        """
        Follow a metapath (list of relation types) from node_id.
        Returns reachable end nodes.
        """
        current = {node_id}
        for rel in metapath:
            next_nodes = set()
            for n in current:
                next_nodes.update(self.get_neighbors_by_relation(n, rel))
            current = next_nodes
        current.discard(node_id)
        return list(current)

    def node_features(self, node_id: str, feature_dim: int = 16) -> np.ndarray:
        """Extract feature vector for a node."""
        ntype = self.G.nodes[node_id].get("ntype", "")
        src = (self.providers if ntype == "Provider" else
               self.services if ntype == "Service" else self.resources)
        entity = src.get(node_id, {})
        fv = entity.get("feature_vector", None)
        if fv:
            return np.array(fv[:feature_dim], dtype=np.float32)
        # Fallback: zeros
        return np.zeros(feature_dim, dtype=np.float32)

    def stats(self) -> dict:
        """Return TKG statistics."""
        from collections import Counter
        edge_types = Counter(data["relation"] for _, _, data in self.G.edges(data=True))
        node_types = Counter(data.get("ntype", "?") for _, data in self.G.nodes(data=True))
        return {
            "num_nodes": self.G.number_of_nodes(),
            "num_edges": self.G.number_of_edges(),
            "node_types": dict(node_types),
            "edge_types": dict(edge_types)
        }

    def to_json(self, path: str):
        """Export TKG structure to JSON file."""
        dirname = os.path.dirname(path)
        if dirname:
            os.makedirs(dirname, exist_ok=True)
        data = {
            "nodes": [{"id": n, **{k: v for k, v in d.items()}}
                      for n, d in self.G.nodes(data=True)],
            "edges": [{"head": u, "tail": v, "relation": d["relation"]}
                      for u, v, d in self.G.edges(data=True)],
            "stats": self.stats()
        }
        with open(path, "w") as f:
            json.dump(data, f, indent=2)
        logger.info(f"TKG exported to {path}")


Writing src/tkg/tkg.py


## 5. Trust meta-path definitions (`src/tkg/metapaths.py`)

In [5]:
%%writefile src/tkg/metapaths.py
"""
metapaths.py - Trust meta-path definitions for the ICPS TKG.
Based on Section 4.2 of the Trust-MPGNN paper.
Author: H. Mezni
"""

# Each metapath is defined as a list of relation types to follow in order.
# Name -> (path of relations, description)

METAPATHS = {
    "PP_trust": {
        "relations": ["TRUST"],
        "node_types": ["Provider", "Provider"],
        "description": "Provider-TRUST-Provider: direct provider trust"
    },
    "PP_trust2": {
        "relations": ["TRUST", "TRUST"],
        "node_types": ["Provider", "Provider", "Provider"],
        "description": "Provider-TRUST-Provider-TRUST-Provider: transitive provider trust"
    },
    "SR_support": {
        "relations": ["SUPPORT"],
        "node_types": ["Service", "Resource"],
        "description": "Service-SUPPORT-Resource: service supports resource usage"
    },
    "SR_oppose": {
        "relations": ["OPPOSE"],
        "node_types": ["Service", "Resource"],
        "description": "Service-OPPOSE-Resource: service opposes resource (conflict)"
    },
    "SS_allied": {
        "relations": ["ALLIED"],
        "node_types": ["Service", "Service"],
        "description": "Service-ALLIED-Service: allied services coalition"
    },
    "SRS": {
        "relations": ["SUPPORT", "SUPPORT"],
        "node_types": ["Service", "Resource", "Service"],
        "description": "Service-SUPPORT-Resource-SUPPORT-Service: mutual resource support"
    },
    "PSR": {
        "relations": ["TRUST", "SUPPORT"],
        "node_types": ["Provider", "Provider", "Resource"],
        "description": "Provider-TRUST-Provider then SERVICE-SUPPORT-Resource trust chain"
    },
    "PPS": {
        "relations": ["TRUST", "TRUST"],
        "node_types": ["Provider", "Provider", "Provider"],
        "description": "Provider-TRUST-Provider-TRUST: extended provider trust"
    },
    "PSS": {
        "relations": ["ALLIED", "ALLIED"],
        "node_types": ["Service", "Service", "Service"],
        "description": "Service-ALLIED-Service-ALLIED-Service: service coalition chain"
    },
    "RR_conflict": {
        "relations": ["CONFLICT"],
        "node_types": ["Resource", "Resource"],
        "description": "Resource-CONFLICT-Resource: resource conflict detection"
    }
}


def get_metapath_names() -> list:
    return list(METAPATHS.keys())


def get_metapath_relations(name: str) -> list:
    return METAPATHS[name]["relations"]


def get_all_relations() -> list:
    """Return flat list of all unique relation sequences used in metapaths."""
    return [v["relations"] for v in METAPATHS.values()]


Writing src/tkg/metapaths.py


## 6. Trust-MPGNN model (`src/gnn/model.py`)
*Fix applied: the metapath attention layers are now fully vectorized (padded neighbor-index tensors + masked batched attention) instead of looping over every node in pure Python — this is the single biggest speedup in the project.*

In [6]:
%%writefile src/gnn/model.py
"""
model.py - Trust-MPGNN: Metapath-guided GNN for trust learning.
Implements Algorithm 1 (metapath-based trust learning) from the paper.

FIXES applied vs the original prototype:
  * MetapathAttentionLayer / TrustMPGNNLayer used a pure-Python `for v in range(N)`
    loop (one tensor op per node, per metapath, per layer, per epoch). This made
    training effectively unusable on anything bigger than a toy graph (a 2000+
    node TKG with the paper's defaults -> ~tens of millions of tiny Python-level
    tensor ops). Rewritten below using padded-neighbor tensors + masked, batched
    attention so the whole layer is a handful of vectorized tensor ops.
Author: H. Mezni (original) / fixes for Colab execution
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import logging

logger = logging.getLogger(__name__)


class MetapathAttentionLayer(nn.Module):
    """
    Single metapath-specific attention aggregation layer (vectorized).
    Computes attention weights e_{vu}^{(m)} and aggregates neighbor embeddings
    for ALL nodes at once using a padded neighbor-index tensor + mask.
    """

    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.3):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        # Attention vector a_m (applied to concatenated source+neighbor projections)
        self.a = nn.Parameter(torch.empty(2 * out_dim))
        nn.init.xavier_uniform_(self.a.view(1, -1))
        self.leaky = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)
        self.out_dim = out_dim

    def forward(self, h: torch.Tensor, nbr_idx: torch.Tensor, nbr_mask: torch.Tensor) -> torch.Tensor:
        """
        h: (N, in_dim) all node embeddings
        nbr_idx: (N, K) long tensor of neighbor indices (padded with 0 where invalid)
        nbr_mask: (N, K) bool tensor, True where the neighbor slot is valid
        Returns: (N, out_dim) aggregated embedding per node (self-projection used
                 for nodes that have no valid neighbors under this metapath).
        """
        N, K = nbr_idx.shape
        Wh = self.W(h)                     # (N, out_dim) - project every node once
        Wu = Wh[nbr_idx]                   # (N, K, out_dim) gather neighbor projections
        Wv_exp = Wh.unsqueeze(1).expand(-1, K, -1)          # (N, K, out_dim)
        concat = torch.cat([Wv_exp, Wu], dim=-1)            # (N, K, 2*out_dim)

        e = self.leaky(concat @ self.a)                     # (N, K)
        e = e.masked_fill(~nbr_mask, float("-inf"))

        has_any = nbr_mask.any(dim=1)                        # (N,)
        # Avoid NaNs from softmax over an all -inf row; replaced afterwards anyway.
        e_safe = torch.where(nbr_mask, e, torch.zeros_like(e))
        alpha = F.softmax(e_safe.masked_fill(~nbr_mask, float("-inf")), dim=1)
        alpha = torch.nan_to_num(alpha, nan=0.0)
        alpha = self.dropout(alpha)

        out = (alpha.unsqueeze(-1) * Wu).sum(dim=1)          # (N, out_dim)
        # Nodes with no valid neighbors fall back to their own projection.
        out = torch.where(has_any.unsqueeze(-1), out, Wh)
        return out


class TrustMPGNNLayer(nn.Module):
    """
    One GNN layer across all metapaths.
    For each metapath m, computes attention-based aggregation, then combines.
    """

    def __init__(self, in_dim: int, out_dim: int, num_metapaths: int, dropout: float = 0.3):
        super().__init__()
        self.mp_layers = nn.ModuleList([
            MetapathAttentionLayer(in_dim, out_dim, dropout) for _ in range(num_metapaths)
        ])
        self.W_self = nn.Linear(in_dim, out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)
        self.act = nn.ELU()

    def forward(self, h: torch.Tensor, nbr_idx_stack: torch.Tensor, nbr_mask_stack: torch.Tensor) -> torch.Tensor:
        """
        h: (N, in_dim) all node embeddings
        nbr_idx_stack: (num_metapaths, N, K) padded neighbor indices
        nbr_mask_stack: (num_metapaths, N, K) validity mask
        Returns updated embeddings (N, out_dim).
        """
        mp_outs = []
        for mp_idx, mp_layer in enumerate(self.mp_layers):
            agg = mp_layer(h, nbr_idx_stack[mp_idx], nbr_mask_stack[mp_idx])
            mp_outs.append(agg)

        aggregated = torch.stack(mp_outs, dim=0).sum(dim=0)  # (N, out_dim)
        h_self = self.W_self(h)
        out = self.act(self.norm(aggregated + h_self))
        return out


class TrustPredictor(nn.Module):
    """
    MLP-based trust predictor: r_hat_uv = sigma(MLP(h_u || h_v))
    """

    def __init__(self, embed_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2 * embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, h_u: torch.Tensor, h_v: torch.Tensor) -> torch.Tensor:
        """Predict trust score for pair (u, v). Shape: scalar or batch."""
        concat = torch.cat([h_u, h_v], dim=-1)
        return self.mlp(concat).squeeze(-1)


class TrustMPGNN(nn.Module):
    """
    Full Trust-MPGNN model: metapath-guided embedding + trust prediction.
    """

    def __init__(self, input_dim: int, embed_dim: int, num_layers: int,
                 num_metapaths: int, dropout: float = 0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, embed_dim)
        self.layers = nn.ModuleList([
            TrustMPGNNLayer(embed_dim, embed_dim, num_metapaths, dropout)
            for _ in range(num_layers)
        ])
        self.predictor = TrustPredictor(embed_dim)

    def forward(self, x: torch.Tensor, nbr_idx_stack: torch.Tensor, nbr_mask_stack: torch.Tensor) -> torch.Tensor:
        """
        x: (N, input_dim) node features
        nbr_idx_stack / nbr_mask_stack: (num_metapaths, N, K) padded neighborhoods
        Returns node embeddings H of shape (N, embed_dim).
        """
        h = F.relu(self.input_proj(x))
        for layer in self.layers:
            h = layer(h, nbr_idx_stack, nbr_mask_stack)
        return h

    def predict_trust(self, h: torch.Tensor, u: int, v: int) -> float:
        """Predict trust score for pair (u, v) given embeddings H."""
        return self.predictor(h[u], h[v]).item()

    def predict_trust_batch(self, h: torch.Tensor,
                             pairs: torch.Tensor) -> torch.Tensor:
        """
        pairs: (B, 2) tensor of (u, v) indices
        Returns (B,) trust scores.
        """
        h_u = h[pairs[:, 0]]
        h_v = h[pairs[:, 1]]
        return self.predictor(h_u, h_v)


Writing src/gnn/model.py


## 7. Metapath-guided neighborhood sampler (`src/gnn/sampler.py`)
*Fix applied: added `to_padded_tensors()`, producing the `(num_metapaths, N, K)` index/mask tensors consumed by the vectorized GNN layers.*

In [7]:
%%writefile src/gnn/sampler.py
"""
sampler.py - Metapath-guided neighborhood sampling for Trust-MPGNN.
Implements the Sample(v, m, G) procedure from Algorithm 1.

FIX: added `to_padded_tensors`, which turns the per-node neighbor-id
neighborhoods into a single (num_metapaths, N, K) index tensor + boolean
mask. This is what lets the GNN layers run as vectorized tensor ops
instead of a Python loop over every node (see src/gnn/model.py).
Author: H. Mezni (original) / fix for Colab execution
"""

import random
import logging
import torch
from typing import List, Dict

logger = logging.getLogger(__name__)


class MetapathSampler:
    """
    Samples neighbors for each node following predefined trust meta-paths.
    N_v^(m) = {u in V | (u,v) in E and u->v follows m in M}
    """

    def __init__(self, tkg, metapaths: dict, sampling_size: int = 20, seed: int = 42):
        """
        tkg: TrustKnowledgeGraph instance
        metapaths: dict {name: {relations: [...]}}
        sampling_size: max neighbors per metapath per node
        """
        self.tkg = tkg
        self.metapaths = metapaths
        self.sampling_size = sampling_size
        random.seed(seed)

    def sample_node(self, node_id: str, mp_relations: List[str]) -> List[str]:
        """
        Follow mp_relations from node_id, return reachable end nodes (sampled).
        """
        current = [node_id]
        for rel in mp_relations:
            next_nodes = []
            for n in current:
                nbrs = self.tkg.get_neighbors_by_relation(n, rel)
                next_nodes.extend(nbrs)
            current = list(set(next_nodes))
            if not current:
                return []

        # Remove self
        if node_id in current:
            current.remove(node_id)

        # Random sampling
        if len(current) > self.sampling_size:
            current = random.sample(current, self.sampling_size)
        return current

    def build_neighborhoods(self) -> Dict[str, Dict[str, List[str]]]:
        """
        Build neighborhoods for ALL nodes across ALL metapaths.
        Returns: {node_id: {mp_name: [neighbor_ids]}}
        """
        logger.info(f"Building metapath neighborhoods for {self.tkg.G.number_of_nodes()} nodes ...")
        neighborhoods = {}
        nodes = list(self.tkg.G.nodes())
        mp_names = list(self.metapaths.keys())

        for i, node_id in enumerate(nodes):
            neighborhoods[node_id] = {}
            for mp_name in mp_names:
                mp_rels = self.metapaths[mp_name]["relations"]
                nbrs = self.sample_node(node_id, mp_rels)
                neighborhoods[node_id][mp_name] = nbrs

            if (i + 1) % 500 == 0:
                logger.info(f"  Sampled {i+1}/{len(nodes)} nodes ...")

        logger.info("Neighborhood sampling complete.")
        return neighborhoods

    def to_index_neighborhoods(self, neighborhoods: Dict[str, Dict[str, List[str]]]) -> list:
        """
        Convert string neighborhoods to integer index neighborhoods.
        Returns list[node_idx][mp_idx] -> list of neighbor node indices.
        (Kept for backward compatibility / inspection; training itself now
        uses `to_padded_tensors` below for speed.)
        """
        node_index = self.tkg.node_index
        mp_names = list(self.metapaths.keys())
        nodes = [self.tkg.index_node[i] for i in range(len(self.tkg.node_index))]

        idx_neighborhoods = []
        for node_id in nodes:
            mp_list = []
            for mp_name in mp_names:
                nbr_ids = neighborhoods.get(node_id, {}).get(mp_name, [])
                nbr_idxs = [node_index[n] for n in nbr_ids if n in node_index]
                mp_list.append(nbr_idxs)
            idx_neighborhoods.append(mp_list)

        return idx_neighborhoods

    def to_padded_tensors(self, neighborhoods: Dict[str, Dict[str, List[str]]]):
        """
        Build padded (num_metapaths, N, K) index + mask tensors from the
        string-keyed neighborhoods, where K = self.sampling_size.

        Returns:
            nbr_idx_stack:  LongTensor (M, N, K) - neighbor node indices
                            (padded with 0; padding is masked out, never used
                            for real aggregation weight).
            nbr_mask_stack: BoolTensor (M, N, K) - True where the slot holds a
                            real neighbor.
        """
        node_index = self.tkg.node_index
        mp_names = list(self.metapaths.keys())
        N = len(node_index)
        K = self.sampling_size
        M = len(mp_names)
        nodes = [self.tkg.index_node[i] for i in range(N)]

        nbr_idx_stack = torch.zeros((M, N, K), dtype=torch.long)
        nbr_mask_stack = torch.zeros((M, N, K), dtype=torch.bool)

        for mp_i, mp_name in enumerate(mp_names):
            for v_idx, node_id in enumerate(nodes):
                nbr_ids = neighborhoods.get(node_id, {}).get(mp_name, [])
                nbr_idxs = [node_index[n] for n in nbr_ids if n in node_index][:K]
                if not nbr_idxs:
                    continue
                cnt = len(nbr_idxs)
                nbr_idx_stack[mp_i, v_idx, :cnt] = torch.tensor(nbr_idxs, dtype=torch.long)
                nbr_mask_stack[mp_i, v_idx, :cnt] = True

        return nbr_idx_stack, nbr_mask_stack


Writing src/gnn/sampler.py


## 8. Trainer — Algorithm 1 (`src/gnn/trainer.py`)
*Fixes applied: single forward pass per epoch (was being recomputed per mini-batch); vectorized `predict_relations`; guard against empty edge-label sets.*

In [8]:
%%writefile src/gnn/trainer.py
"""
trainer.py - Trust learning trainer implementing Algorithm 1 from the paper.
Manages embedding learning, trust prediction, and saving results.

FIXES applied vs the original prototype:
  * The original training loop called `self.model(X, neighborhoods_idx)`
    INSIDE the mini-batch loop, i.e. it recomputed the *entire* forward pass
    (every node, every metapath, every layer) once per mini-batch instead of
    once per epoch. For a full-batch/transductive GNN like this one, that
    makes training cost scale with (num_pairs / batch_size) extra forward
    passes for no benefit. Fixed to do exactly one forward pass per epoch.
  * `predict_relations` looped over every edge in Python calling
    `model.predict_trust` one pair at a time. Replaced with a single
    vectorized `predict_trust_batch` call.
Author: H. Mezni (original) / fixes for Colab execution
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import logging
import os
from typing import Tuple, Dict, List

logger = logging.getLogger(__name__)


class TrustTrainer:
    """
    Orchestrates Algorithm 1:
    1. Initialize node embeddings from features
    2. Metapath-guided neighborhood sampling (done by MetapathSampler)
    3. Attention-based aggregation (GNN layers)
    4. Trust prediction and classification (Delta/Gamma sets)
    """

    def __init__(self, model, tkg, sampler, config: dict):
        self.model = model
        self.tkg = tkg
        self.sampler = sampler
        self.config = config
        self.embed_dim = config.get("embed_dim", 128)
        self.lr = config.get("learning_rate", 0.001)
        self.weight_decay = config.get("weight_decay", 1e-4)
        self.epochs = config.get("epochs", 200)
        self.batch_size = config.get("batch_size", 256)
        self.theta = config.get("trust_threshold", 0.6)
        self.optimizer = optim.Adam(model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        self.loss_fn = nn.BCELoss()
        self.device = torch.device("cpu")

    def build_features(self, feature_dim: int = 16) -> torch.Tensor:
        """Build feature matrix X from TKG node features."""
        nodes = [self.tkg.index_node[i] for i in range(len(self.tkg.node_index))]
        X = np.stack([self.tkg.node_features(n, feature_dim) for n in nodes])
        return torch.tensor(X, dtype=torch.float32)

    def build_edge_labels(self) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Build positive (trusted) and negative (conflict) edge samples for training.
        Positive: TRUST, SUPPORT, ALLIED edges -> label 1
        Negative: OPPOSE, CONFLICT edges -> label 0
        """
        pos_pairs, neg_pairs = [], []
        node_index = self.tkg.node_index

        for u, v, data in self.tkg.G.edges(data=True):
            if u not in node_index or v not in node_index:
                continue
            ui, vi = node_index[u], node_index[v]
            rel = data.get("relation", "")
            if rel in ("TRUST", "SUPPORT", "ALLIED"):
                pos_pairs.append((ui, vi))
            elif rel in ("OPPOSE", "CONFLICT", "NEUTRAL"):
                neg_pairs.append((ui, vi))

        # Balance: sample same number of negatives as positives
        min_size = min(len(pos_pairs), len(neg_pairs))
        if min_size == 0:
            min_size = max(len(pos_pairs), len(neg_pairs))

        import random
        random.shuffle(pos_pairs)
        random.shuffle(neg_pairs)
        pos_pairs = pos_pairs[:min_size]
        neg_pairs = neg_pairs[:min_size]

        pairs = pos_pairs + neg_pairs
        labels = [1.0] * len(pos_pairs) + [0.0] * len(neg_pairs)

        if not pairs:
            # Degenerate graph with no usable edges - avoid a crash downstream.
            return torch.zeros((0, 2), dtype=torch.long), torch.zeros((0,), dtype=torch.float32)

        pairs_t = torch.tensor(pairs, dtype=torch.long)
        labels_t = torch.tensor(labels, dtype=torch.float32)
        return pairs_t, labels_t

    def train(self, nbr_idx_stack: torch.Tensor, nbr_mask_stack: torch.Tensor) -> torch.Tensor:
        """
        Run the full training loop (Algorithm 1, lines 1-13).
        One forward pass per epoch (full-batch / transductive training);
        the edge-label pairs are only used to compute the loss, never to
        trigger another forward pass.

        nbr_idx_stack / nbr_mask_stack: (num_metapaths, N, K) padded
            neighborhoods, as produced by MetapathSampler.to_padded_tensors.
        Returns: final trust embedding H (N, embed_dim)
        """
        X = self.build_features(self.config.get("feature_dim", 16))
        pairs, labels = self.build_edge_labels()
        N = X.shape[0]

        logger.info(f"Training TrustMPGNN: {N} nodes, {len(pairs)} edge samples, {self.epochs} epochs ...")

        if len(pairs) == 0:
            logger.warning("No labeled edges available for training; returning untrained embeddings.")
            self.model.eval()
            with torch.no_grad():
                return self.model(X, nbr_idx_stack, nbr_mask_stack)

        best_loss = float("inf")
        best_H = None

        self.model.train()
        for epoch in range(1, self.epochs + 1):
            self.optimizer.zero_grad()

            # Single forward pass for the whole graph this epoch.
            H = self.model(X, nbr_idx_stack, nbr_mask_stack)

            # Mini-batch the LOSS (not the forward pass) for memory friendliness
            # on very large pair sets, but reuse the same H/graph throughout.
            perm = torch.randperm(len(pairs))
            pairs_shuf = pairs[perm]
            labels_shuf = labels[perm]

            total_loss = 0.0
            n_chunks = 0
            for start in range(0, len(pairs_shuf), self.batch_size):
                batch_pairs = pairs_shuf[start:start + self.batch_size]
                batch_labels = labels_shuf[start:start + self.batch_size]
                scores = self.model.predict_trust_batch(H, batch_pairs)
                loss = self.loss_fn(scores, batch_labels)
                total_loss = total_loss + loss
                n_chunks += 1

            avg_loss = total_loss / max(n_chunks, 1)
            avg_loss.backward()
            self.optimizer.step()

            loss_val = avg_loss.item()
            if loss_val < best_loss:
                best_loss = loss_val
                best_H = H.detach().clone()

            if epoch % max(1, self.epochs // 10) == 0 or epoch == 1:
                logger.info(f"  Epoch {epoch}/{self.epochs} | Loss: {loss_val:.4f}")

        logger.info(f"Training complete. Best loss: {best_loss:.4f}")
        return best_H

    def predict_relations(self, H: torch.Tensor) -> Tuple[list, list, list]:
        """
        Algorithm 1, lines 15-25: predict trust/conflict for all node pairs.
        Uses candidate pairs from existing edges only (for efficiency).
        Returns: E_hat (all predicted), Delta (trusted), Gamma (conflict).
        Vectorized: one batched forward through the predictor instead of a
        Python loop calling `predict_trust` edge-by-edge.
        """
        self.model.eval()
        node_index = self.tkg.node_index

        edges = [(u, v) for u, v, data in self.tkg.G.edges(data=True)
                 if u in node_index and v in node_index]
        if not edges:
            return [], [], []

        pairs = torch.tensor([[node_index[u], node_index[v]] for u, v in edges], dtype=torch.long)

        with torch.no_grad():
            scores = self.model.predict_trust_batch(H, pairs).cpu().numpy()

        E_hat, Delta, Gamma = [], [], []
        for (u, v), score in zip(edges, scores):
            score = float(score)
            triple = (u, score, v)
            E_hat.append(triple)
            if score >= self.theta:
                Delta.append(triple)
            else:
                Gamma.append(triple)

        logger.info(f"Trust prediction: {len(Delta)} trusted, {len(Gamma)} conflict pairs (theta={self.theta})")
        return E_hat, Delta, Gamma

    def save_embeddings(self, H: torch.Tensor, path: str):
        """Save trust embedding space to file."""
        dirname = os.path.dirname(path)
        if dirname:
            os.makedirs(dirname, exist_ok=True)
        torch.save({"embeddings": H, "node_index": self.tkg.node_index}, path)
        logger.info(f"Embeddings saved to {path}")


Writing src/gnn/trainer.py


## 9. IoT service composer — Algorithm 2 (`src/composition/composer.py`)

In [9]:
%%writefile src/composition/composer.py
"""
composer.py - Trust-aware IoT service selection and composition.
Implements Algorithm 2 from the Trust-MPGNN paper.
Author: H. Mezni
"""

import torch
import numpy as np
import logging
import json
from typing import List, Dict, Tuple, Optional

logger = logging.getLogger(__name__)


def cosine_similarity_matrix(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """
    Compute cosine similarity matrix between rows of A and B.
    S[i,j] = (A[i] . B[j]) / (||A[i]|| ||B[j]||)
    """
    A_norm = A / (A.norm(dim=1, keepdim=True) + 1e-8)
    B_norm = B / (B.norm(dim=1, keepdim=True) + 1e-8)
    return A_norm @ B_norm.T  # (|A|, |B|)


def find_best_service(H: torch.Tensor, F_S: torch.Tensor, task_vec: torch.Tensor,
                       trusted_service_idxs: List[int],
                       used_idxs: Optional[set] = None) -> int:
    """
    Find best IoT service for a task using cosine similarity between task_vec and service embeddings.
    Excludes already-used indices to promote diversity across workflow tasks.
    Returns global node index.
    """
    if len(trusted_service_idxs) == 0:
        return -1
    # Prefer unused candidates
    candidates = [i for i in trusted_service_idxs if used_idxs is None or i not in used_idxs]
    if not candidates:
        candidates = trusted_service_idxs

    H_candidates = H[candidates]                  # (n_c, embed_dim)
    embed_dim = H_candidates.shape[1]
    # Project task_vec to embed_dim if needed
    if task_vec.shape[0] != embed_dim:
        # Repeat/truncate to match embed_dim
        repeats = (embed_dim + task_vec.shape[0] - 1) // task_vec.shape[0]
        task_proj = task_vec.repeat(repeats)[:embed_dim]
    else:
        task_proj = task_vec
    task_norm = task_proj / (task_proj.norm() + 1e-8)
    H_norm = H_candidates / (H_candidates.norm(dim=1, keepdim=True) + 1e-8)
    scores = H_norm @ task_norm                   # (n_c,)
    best_local = scores.argmax().item()
    return candidates[best_local]


def find_best_resource(H: torch.Tensor, F_R: torch.Tensor, res_vec: torch.Tensor,
                        trusted_resource_idxs: List[int],
                        used_idxs: Optional[set] = None) -> int:
    """
    Find best IoT resource using cosine similarity. Excludes already-used resources.
    """
    if len(trusted_resource_idxs) == 0:
        return -1
    candidates = [i for i in trusted_resource_idxs if used_idxs is None or i not in used_idxs]
    if not candidates:
        candidates = trusted_resource_idxs

    H_candidates = H[candidates]
    embed_dim = H_candidates.shape[1]
    if res_vec.shape[0] != embed_dim:
        repeats = (embed_dim + res_vec.shape[0] - 1) // res_vec.shape[0]
        res_proj = res_vec.repeat(repeats)[:embed_dim]
    else:
        res_proj = res_vec
    res_norm = res_proj / (res_proj.norm() + 1e-8)
    H_norm = H_candidates / (H_candidates.norm(dim=1, keepdim=True) + 1e-8)
    scores = H_norm @ res_norm
    best_local = scores.argmax().item()
    return candidates[best_local]


class IoTComposer:
    """
    Trust-aware IoT service composition module.
    Uses precomputed trust embeddings H to select and compose IoT services/resources.
    """

    def __init__(self, tkg, H: torch.Tensor, Delta: list, Gamma: list, config: dict):
        """
        tkg: TrustKnowledgeGraph
        H: (N, d) trust embedding matrix
        Delta: trusted relation triples (u, score, v)
        Gamma: conflict relation triples
        config: composition configuration
        """
        self.tkg = tkg
        self.H = H
        self.Delta = Delta
        self.Gamma = Gamma
        self.config = config
        self.conflict_set = {(u, v) for u, score, v in Gamma}
        self.trust_set = {(u, v) for u, score, v in Delta}

        # Pre-sort node indices by type
        self.service_idxs = [tkg.node_index[sid] for sid in tkg.services
                              if sid in tkg.node_index]
        self.resource_idxs = [tkg.node_index[rid] for rid in tkg.resources
                               if rid in tkg.node_index]
        self.provider_idxs = [tkg.node_index[pid] for pid in tkg.providers
                               if pid in tkg.node_index]

        # Build trusted subsets (filter out high-conflict entities)
        self.trusted_service_idxs = self._filter_trusted(self.service_idxs)
        self.trusted_resource_idxs = self._filter_trusted(self.resource_idxs)

    def _filter_trusted(self, idxs: List[int]) -> List[int]:
        """
        Keep only entities that appear in Delta and NOT in Gamma conflict pairs.
        """
        index_node = self.tkg.index_node
        trusted = []
        for idx in idxs:
            node_id = index_node.get(idx)
            if node_id is None:
                continue
            in_conflict = any(
                (node_id == u or node_id == v) for u, v in self.conflict_set
            )
            if not in_conflict:
                trusted.append(idx)
        return trusted if trusted else idxs  # fallback to all if none remain

    def _proximity_degree(self, service_idx: int, resource_idx: int) -> float:
        """
        Compute proximity degree d: combines trust chain ratio with embedding similarity.
        d is used in the composition scoring function (Eq. score).
        """
        s_id = self.tkg.index_node.get(service_idx, "")
        r_id = self.tkg.index_node.get(resource_idx, "")
        trust_count = sum(1 for u, v in self.trust_set
                          if u in (s_id, r_id) or v in (s_id, r_id))
        conflict_count = sum(1 for u, v in self.conflict_set
                             if u in (s_id, r_id) or v in (s_id, r_id))
        total = trust_count + conflict_count
        if total == 0:
            # Fallback: use cosine similarity between embeddings
            if service_idx < self.H.shape[0] and resource_idx < self.H.shape[0]:
                hs = self.H[service_idx]
                hr = self.H[resource_idx]
                cos = float((hs @ hr) / (hs.norm() * hr.norm() + 1e-8))
                return max(0.0, min(1.0, (cos + 1.0) / 2.0))
            return 0.5
        return trust_count / total

    def compose(self, workflow: List[Dict]) -> Dict:
        """
        Algorithm 2: Trust-aware IoT Service Selection.
        workflow: list of {task_id, task_name, task_features, resource_features}
        Returns: composition result with assigned services, resources, and score.
        """
        logger.info(f"Composing IoT application for workflow of {len(workflow)} tasks ...")
        V_prime = []  # Output: assigned (service, resource) pairs

        # Feature matrices for cosine similarity
        if len(self.trusted_service_idxs) > 0:
            F_S = self.H[self.trusted_service_idxs]
        else:
            F_S = self.H[self.service_idxs]

        if len(self.trusted_resource_idxs) > 0:
            F_R = self.H[self.trusted_resource_idxs]
        else:
            F_R = self.H[self.resource_idxs]

        used_s, used_r = set(), set()
        for task in workflow:
            task_feat = torch.tensor(task.get("task_features", [0.0] * 16), dtype=torch.float32)
            res_feat = torch.tensor(task.get("resource_features", [0.0] * 16), dtype=torch.float32)

            # find_best_service (exclude already-used for diversity)
            s_idx = find_best_service(self.H, F_S, task_feat, self.trusted_service_idxs, used_s)
            # find_best_resource
            r_idx = find_best_resource(self.H, F_R, res_feat, self.trusted_resource_idxs, used_r)
            if s_idx >= 0:
                used_s.add(s_idx)
            if r_idx >= 0:
                used_r.add(r_idx)

            s_id = self.tkg.index_node.get(s_idx, "UNKNOWN")
            r_id = self.tkg.index_node.get(r_idx, "UNKNOWN")
            s_meta = self.tkg.services.get(s_id, {})
            r_meta = self.tkg.resources.get(r_id, {})

            d = self._proximity_degree(s_idx, r_idx) if s_idx >= 0 and r_idx >= 0 else 0.5

            V_prime.append({
                "task_id": task.get("task_id"),
                "task_name": task.get("task_name"),
                "service_id": s_id,
                "service_name": s_meta.get("name", ""),
                "resource_id": r_id,
                "resource_name": r_meta.get("name", ""),
                "proximity_degree": round(d, 4),
                "qos": s_meta.get("qos", {})
            })

        # Evaluate composition (Eq. score)
        score = self._composition_score(V_prime)

        result = {
            "workflow_size": len(workflow),
            "assignments": V_prime,
            "composition_score": round(score, 4),
            "num_services": len({a["service_id"] for a in V_prime}),
            "num_resources": len({a["resource_id"] for a in V_prime}),
            "trust_summary": {
                "trusted_set_size": len(self.trusted_service_idxs),
                "conflict_set_size": len(self.conflict_set),
                "trust_set_size": len(self.trust_set)
            }
        }
        return result

    def _composition_score(self, assignments: list) -> float:
        """
        Eq. Score(W^p) = d/(1-d) * 1/(|S'|*|R'|) * sum(QoS_i * |S_i in W^p|)
        """
        if not assignments:
            return 0.0
        unique_services = {a["service_id"] for a in assignments}
        unique_resources = {a["resource_id"] for a in assignments}
        S_prime = len(unique_services)
        R_prime = len(unique_resources)

        # Average proximity degree
        d = np.mean([a["proximity_degree"] for a in assignments])
        if d >= 1.0:
            d = 0.99
        conflict_factor = 1 - d
        if conflict_factor < 1e-6:
            conflict_factor = 1e-6

        # QoS sum: use reliability as primary QoS metric
        qos_sum = 0.0
        for a in assignments:
            qos = a.get("qos", {})
            reliability = qos.get("reliability", 0.5)
            count = sum(1 for b in assignments if b["service_id"] == a["service_id"])
            qos_sum += reliability * count

        score = (d / conflict_factor) * (1.0 / max(S_prime * R_prime, 1)) * qos_sum
        return float(score)

    def conflict_severity(self, assignments: list) -> float:
        """
        Compute conflict severity = 1 - average(proximity_degree).
        """
        if not assignments:
            return 1.0
        d = np.mean([a["proximity_degree"] for a in assignments])
        return round(1.0 - d, 4)

    def trust_score(self, assignments: list) -> float:
        """Average proximity (trust) score of the composition."""
        if not assignments:
            return 0.0
        return round(float(np.mean([a["proximity_degree"] for a in assignments])), 4)


Writing src/composition/composer.py


## 10. Workflow / query representation (`src/composition/workflow.py`)

In [10]:
%%writefile src/composition/workflow.py
"""
workflow.py - IoT workflow representation and query parsing.
Defines the abstract IoT workflow W_u and user query format.
Author: H. Mezni
"""

import json
import random
import numpy as np
from typing import List, Dict


def random_feature_vector(dim: int = 16) -> List[float]:
    """Generate a random normalized feature vector."""
    v = np.random.uniform(0, 1, dim)
    v = v / (np.linalg.norm(v) + 1e-8)
    return v.tolist()


def create_workflow(tasks: List[Dict]) -> List[Dict]:
    """
    Create an abstract IoT workflow from a list of task descriptors.
    Each task: {task_id, task_name, task_features (optional), resource_features (optional)}
    """
    workflow = []
    for i, task in enumerate(tasks):
        workflow.append({
            "task_id": task.get("task_id", f"T{i+1}"),
            "task_name": task.get("task_name", f"Task_{i+1}"),
            "task_features": task.get("task_features", random_feature_vector(16)),
            "resource_features": task.get("resource_features", random_feature_vector(16))
        })
    return workflow


def load_workflow_from_file(path: str) -> List[Dict]:
    """Load a workflow from a JSON file."""
    with open(path) as f:
        data = json.load(f)
    return data.get("workflow", data)


def generate_workflow(size: int, seed: int = 42) -> List[Dict]:
    """Generate a synthetic IoT workflow of given size."""
    random.seed(seed)
    np.random.seed(seed)
    task_templates = [
        "HealthMonitor", "TemperatureControl", "AccessControl", "EnergyManage",
        "SecurityMonitor", "LightControl", "OccupancyDetect", "AirQuality",
        "TrafficMonitor", "ParkingManage", "WasteManage", "WaterMonitor",
        "ProductionTrack", "InventoryManage", "PatientMonitor", "FitnessTrack"
    ]
    tasks = []
    for i in range(size):
        name = f"{random.choice(task_templates)}_{i+1}"
        tasks.append({
            "task_id": f"T{i+1:02d}",
            "task_name": name,
            "task_features": random_feature_vector(16),
            "resource_features": random_feature_vector(16)
        })
    return create_workflow(tasks)


# --- Predefined test queries ---
SAMPLE_QUERIES = [
    {
        "query_id": "Q1",
        "description": "Smart healthcare monitoring application",
        "workflow": [
            {"task_id": "T1", "task_name": "VitalSigns", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T2", "task_name": "MedicationTracker", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T3", "task_name": "FitnessTracker", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)}
        ]
    },
    {
        "query_id": "Q2",
        "description": "Smart building energy management",
        "workflow": [
            {"task_id": "T1", "task_name": "EnergyMonitor", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T2", "task_name": "ClimateControl", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T3", "task_name": "LightControl", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T4", "task_name": "OccupancyDetect", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)}
        ]
    },
    {
        "query_id": "Q3",
        "description": "Smart mobility and parking",
        "workflow": [
            {"task_id": "T1", "task_name": "ParkingMonitor", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T2", "task_name": "TrafficControl", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T3", "task_name": "EVCharging", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)}
        ]
    },
    {
        "query_id": "Q4",
        "description": "Smart home automation",
        "workflow": [
            {"task_id": "T1", "task_name": "SecurityCam", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T2", "task_name": "SmartLock", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T3", "task_name": "AmbientLight", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T4", "task_name": "ThermostatCtrl", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)},
            {"task_id": "T5", "task_name": "FloodDetect", "task_features": random_feature_vector(16), "resource_features": random_feature_vector(16)}
        ]
    }
]


Writing src/composition/workflow.py


## 11. Baseline methods — Trust-GNN, FFCA-IoTSC, TQoSC, GNN-IoTSC (`src/composition/baselines.py`)

In [11]:
%%writefile src/composition/baselines.py
"""
baselines.py - Baseline composition methods for comparison experiments.
Implements Trust-GNN, FFCA-IoTSC, TQoSC, and GNN-IoTSC baselines.
Author: H. Mezni
"""

import numpy as np
import torch
import random
import logging
from typing import List, Dict

logger = logging.getLogger(__name__)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) + 1e-8) * (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a, b) / denom)


class TrustGNN:
    """
    Baseline: Standard GNN without metapath guidance.
    Uses mean pooling over all neighbor embeddings (no semantic filtering).
    """

    def __init__(self, H_mean: np.ndarray, tkg, Gamma_ids: set):
        self.H = H_mean
        self.tkg = tkg
        self.Gamma_ids = Gamma_ids  # conflict node ids

    def compose(self, workflow: List[Dict]) -> Dict:
        """Select services using unguided embeddings (mean pooling)."""
        assignments = []
        service_ids = [sid for sid in self.tkg.services if sid not in self.Gamma_ids]
        resource_ids = [rid for rid in self.tkg.resources if rid not in self.Gamma_ids]

        for task in workflow:
            tf = np.array(task.get("task_features", [0.0] * 16))
            rf = np.array(task.get("resource_features", [0.0] * 16))

            best_s = max(service_ids,
                         key=lambda sid: cosine_sim(self.H[self.tkg.node_index[sid]], tf)
                         if sid in self.tkg.node_index else 0, default=None)
            best_r = max(resource_ids,
                         key=lambda rid: cosine_sim(self.H[self.tkg.node_index[rid]], rf)
                         if rid in self.tkg.node_index else 0, default=None)

            d = random.uniform(0.65, 0.92)  # simulated proximity for standard GNN
            s_meta = self.tkg.services.get(best_s, {})
            r_meta = self.tkg.resources.get(best_r, {})
            assignments.append({
                "task_id": task["task_id"],
                "service_id": best_s, "service_name": s_meta.get("name", ""),
                "resource_id": best_r, "resource_name": r_meta.get("name", ""),
                "proximity_degree": d,
                "qos": s_meta.get("qos", {})
            })
        return {"workflow_size": len(workflow), "assignments": assignments}


class FFCAIoTSC:
    """
    Baseline: Fuzzy conflict analysis-based IoT service composition.
    Uses situation tables and fuzzy scoring (simulated).
    """

    def __init__(self, tkg, conflict_density: float = 0.2):
        self.tkg = tkg
        self.conflict_density = conflict_density

    def compose(self, workflow: List[Dict]) -> Dict:
        """Fuzzy conflict-aware composition (simplified simulation)."""
        assignments = []
        service_ids = list(self.tkg.services.keys())
        resource_ids = list(self.tkg.resources.keys())

        for task in workflow:
            s_id = random.choice(service_ids)
            r_id = random.choice(resource_ids)
            # Fuzzy score degrades with conflict density
            d = max(0.4, random.uniform(0.55, 0.88) - self.conflict_density * 0.3)
            s_meta = self.tkg.services.get(s_id, {})
            r_meta = self.tkg.resources.get(r_id, {})
            assignments.append({
                "task_id": task["task_id"],
                "service_id": s_id, "service_name": s_meta.get("name", ""),
                "resource_id": r_id, "resource_name": r_meta.get("name", ""),
                "proximity_degree": d,
                "qos": s_meta.get("qos", {})
            })
        return {"workflow_size": len(workflow), "assignments": assignments}


class TQoSC:
    """
    Baseline: QoS-only composition without trust or conflict awareness.
    Selects services with highest QoS regardless of trust.
    """

    def __init__(self, tkg, qos_weights: dict = None):
        self.tkg = tkg
        self.qos_weights = qos_weights or {"reliability": 0.4, "response_time": 0.3, "cost": 0.3}

    def _qos_score(self, entity: dict) -> float:
        qos = entity.get("qos", {})
        r = qos.get("reliability", 0.5) * self.qos_weights.get("reliability", 0.4)
        rt = (1 - min(qos.get("response_time", 300) / 500.0, 1.0)) * self.qos_weights.get("response_time", 0.3)
        c = (1 - min(qos.get("cost", 5.0) / 10.0, 1.0)) * self.qos_weights.get("cost", 0.3)
        return r + rt + c

    def compose(self, workflow: List[Dict]) -> Dict:
        """Select best QoS services, ignoring trust."""
        assignments = []
        sorted_services = sorted(self.tkg.services.values(), key=self._qos_score, reverse=True)
        sorted_resources = sorted(self.tkg.resources.values(), key=self._qos_score, reverse=True)

        for i, task in enumerate(workflow):
            s = sorted_services[i % len(sorted_services)]
            r = sorted_resources[i % len(sorted_resources)]
            d = random.uniform(0.30, 0.65)  # low trust (not trust-aware)
            assignments.append({
                "task_id": task["task_id"],
                "service_id": s["id"], "service_name": s.get("name", ""),
                "resource_id": r["id"], "resource_name": r.get("name", ""),
                "proximity_degree": d,
                "qos": s.get("qos", {})
            })
        return {"workflow_size": len(workflow), "assignments": assignments}


class GNNIoTSC:
    """
    Baseline: GNN without metapath or conflict-aware filtering (simplified Trust-MPGNN).
    """

    def __init__(self, H: np.ndarray, tkg):
        self.H = H
        self.tkg = tkg

    def compose(self, workflow: List[Dict]) -> Dict:
        """GNN-based composition without metapath filtering."""
        assignments = []
        service_ids = list(self.tkg.services.keys())
        resource_ids = list(self.tkg.resources.keys())

        for task in workflow:
            tf = np.array(task.get("task_features", [0.0] * 16))
            rf = np.array(task.get("resource_features", [0.0] * 16))

            best_s = max(service_ids,
                         key=lambda sid: cosine_sim(self.H[self.tkg.node_index[sid]], tf)
                         if sid in self.tkg.node_index else 0, default=None)
            best_r = max(resource_ids,
                         key=lambda rid: cosine_sim(self.H[self.tkg.node_index[rid]], rf)
                         if rid in self.tkg.node_index else 0, default=None)

            d = random.uniform(0.50, 0.85)
            s_meta = self.tkg.services.get(best_s, {})
            r_meta = self.tkg.resources.get(best_r, {})
            assignments.append({
                "task_id": task["task_id"],
                "service_id": best_s, "service_name": s_meta.get("name", ""),
                "resource_id": best_r, "resource_name": r_meta.get("name", ""),
                "proximity_degree": d,
                "qos": s_meta.get("qos", {})
            })
        return {"workflow_size": len(workflow), "assignments": assignments}


def compute_metrics(result: dict, conflict_set: set, tkg) -> dict:
    """
    Compute evaluation metrics from a composition result:
    - success_rate: fraction of tasks with valid (non-conflicting) assignments
    - trust_score: average proximity degree
    - conflict_severity: 1 - trust_score
    """
    assignments = result.get("assignments", [])
    if not assignments:
        return {"success_rate": 0.0, "trust_score": 0.0, "conflict_severity": 1.0,
                "services_used": 0, "resources_used": 0}

    valid = 0
    for a in assignments:
        sid = a.get("service_id", "")
        rid = a.get("resource_id", "")
        if (sid, rid) not in conflict_set and sid and rid:
            valid += 1

    success_rate = valid / len(assignments)
    trust_score = np.mean([a.get("proximity_degree", 0.5) for a in assignments])
    conflict_severity = 1.0 - trust_score
    services_used = len({a["service_id"] for a in assignments})
    resources_used = len({a["resource_id"] for a in assignments})

    return {
        "success_rate": round(float(success_rate), 4),
        "trust_score": round(float(trust_score), 4),
        "conflict_severity": round(float(conflict_severity), 4),
        "services_used": services_used,
        "resources_used": resources_used
    }


Writing src/composition/baselines.py


## 12. ICPS dataset generator (`data/gen_dataset.py`)

In [12]:
%%writefile data/gen_dataset.py
"""
gen_dataset.py - Generate the hybrid ICPS dataset (IoT services, resources, providers, trust/conflict relations)
Author: H. Mezni
"""

import json
import random
import uuid
import os
import sys
import logging
import argparse

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# IoT domain knowledge
PROTOCOLS = ["Zigbee", "Bluetooth 5.0", "Wi-Fi 6", "Z-Wave", "MQTT", "CoAP", "LoRaWAN", "NB-IoT"]
DATA_FORMATS = ["JSON", "XML", "FHIR", "HL7", "CSV", "Protobuf", "CBOR"]
PRIVACY_POLICIES = ["GDPR", "HIPAA", "CCPA", "ISO27001", "NIST"]
CATEGORIES = ["smart_home", "smart_health", "smart_mobility", "smart_energy",
              "smart_security", "smart_education", "smart_retail", "smart_agriculture"]
SERVICE_TYPES = [
    "HealthTracker", "TemperatureMonitor", "OccupancyDetector", "EnergyManager",
    "SecurityCamera", "SmartLock", "LightController", "ClimateControl",
    "ParkingManager", "TrafficMonitor", "WasteManager", "WaterMonitor",
    "AirQualityMonitor", "NoiseMonitor", "ProductionTracker", "InventoryManager",
    "PatientMonitor", "MedicationTracker", "VitalSignsMonitor", "FitnessTracker",
    "SmartMeter", "EVCharger", "SolarMonitor", "HVACController",
    "AccessControl", "IntruderDetector", "FireDetector", "FloodSensor",
    "LearningAssistant", "AttendanceTracker", "SmartWhiteBoard", "ProjectorControl",
    "CheckoutMonitor", "StockTracker", "CustomerFlow", "PointOfSale",
    "IrrigationControl", "SoilMonitor", "GreenHouseControl", "LivestockTracker"
]
RESOURCE_TYPES = [
    "TemperatureSensor", "HumiditySensor", "MotionSensor", "OccupancySensor",
    "LightSensor", "SoundSensor", "PressureSensor", "ProximitySensor",
    "CameraDevice", "MicrophoneDevice", "SpeakerDevice", "DisplayDevice",
    "ActuatorDevice", "RelayDevice", "BuzzerDevice", "LEDStrip",
    "WearableDevice", "BioSensor", "Accelerometer", "GPS_Module",
    "SmartMeter_Res", "BatteryPack", "SolarPanel", "PowerGrid",
    "EdgeNode", "FogNode", "CloudGateway", "WiFiRouter",
    "SmartLock_Res", "DoorSensor", "WindowSensor", "SirenDevice",
    "SmartBoard", "Projector", "AudioHub", "MainScreen",
    "BarCodeScanner", "RFID_Reader", "PrinterDevice", "NFCReader",
    "IrrigationValve", "SoilProbe", "WeatherStation", "DroneController"
]
PROVIDER_NAMES = [
    "TechCorp", "SmartSys", "IoTHub", "ConnectPro", "SensorNet",
    "CloudIoT", "EdgeTech", "FogSystems", "SmartCity", "GreenTech",
    "HealthNet", "MobileSmart", "SecureIoT", "EduTech", "RetailSmart",
    "AgroSmart", "EnergySmart", "SafeNet", "DataStream", "OmniSense"
]


def generate_providers(n: int) -> list:
    """Generate IoT service providers with trust metadata."""
    providers = []
    for i in range(n):
        p = {
            "id": f"P{i:04d}",
            "name": f"{random.choice(PROVIDER_NAMES)}_{i}",
            "type": "Provider",
            "category": random.choice(CATEGORIES),
            "rating": round(random.uniform(2.5, 5.0), 2),
            "num_reviews": random.randint(10, 5000),
            "location": {"city": f"City_{i % 20}", "country": "TN"},
            "protocols": random.sample(PROTOCOLS, random.randint(1, 3)),
            "privacy_compliance": random.sample(PRIVACY_POLICIES, random.randint(1, 2)),
            "years_active": random.randint(1, 15)
        }
        providers.append(p)
    return providers


def generate_services(n: int, providers: list) -> list:
    """Generate IoT services with functional and QoS attributes."""
    services = []
    for i in range(n):
        provider = random.choice(providers)
        stype = random.choice(SERVICE_TYPES)
        s = {
            "id": f"S{i:05d}",
            "name": f"{stype}_{i}",
            "type": "Service",
            "category": provider["category"],
            "provider_id": provider["id"],
            "capabilities": random.sample(SERVICE_TYPES, random.randint(1, 4)),
            "protocols": random.sample(PROTOCOLS, random.randint(1, 3)),
            "data_format": random.choice(DATA_FORMATS),
            "privacy_compliance": random.sample(PRIVACY_POLICIES, random.randint(1, 2)),
            "qos": {
                "reliability": round(random.uniform(0.7, 1.0), 3),
                "response_time": round(random.uniform(10, 500), 1),
                "cost": round(random.uniform(0.01, 10.0), 3),
                "availability": round(random.uniform(0.9, 1.0), 3),
                "throughput": round(random.uniform(10, 1000), 1)
            },
            "feature_vector": [round(random.uniform(0, 1), 4) for _ in range(16)]
        }
        services.append(s)
    return services


def generate_resources(n: int, providers: list) -> list:
    """Generate IoT resources with physical and QoS attributes."""
    resources = []
    for i in range(n):
        provider = random.choice(providers)
        rtype = random.choice(RESOURCE_TYPES)
        r = {
            "id": f"R{i:05d}",
            "name": f"{rtype}_{i}",
            "type": "Resource",
            "resource_type": rtype,
            "provider_id": provider["id"],
            "protocols": random.sample(PROTOCOLS, random.randint(1, 3)),
            "data_format": random.choice(DATA_FORMATS),
            "privacy_compliance": random.sample(PRIVACY_POLICIES, random.randint(0, 2)),
            "energy_profile": round(random.uniform(0.1, 10.0), 2),
            "sensing_range": round(random.uniform(1, 100), 1),
            "qos": {
                "reliability": round(random.uniform(0.7, 1.0), 3),
                "response_time": round(random.uniform(5, 200), 1),
                "cost": round(random.uniform(0.001, 5.0), 3),
                "energy_efficiency": round(random.uniform(0.5, 1.0), 3)
            },
            "feature_vector": [round(random.uniform(0, 1), 4) for _ in range(16)]
        }
        resources.append(r)
    return resources


def is_compatible(e1: dict, e2: dict) -> bool:
    """Check protocol/format compatibility between two entities."""
    shared_protocols = set(e1.get("protocols", [])) & set(e2.get("protocols", []))
    shared_privacy = set(e1.get("privacy_compliance", [])) & set(e2.get("privacy_compliance", []))
    return len(shared_protocols) > 0 or len(shared_privacy) > 0


def generate_relations(providers: list, services: list, resources: list,
                        target_total: int, conflict_density: float) -> list:
    """
    Generate trust and conflict relations among ICPS entities.
    Relation types: TRUST (P-P), SUPPORT/OPPOSE/NEUTRAL (S-R), ALLIED (S-S), CONFLICT (R-R).
    conflict_density controls percentage of OPPOSE/CONFLICT edges.
    """
    relations = []
    pid = {p["id"]: p for p in providers}
    sid = {s["id"]: s for s in services}
    rid = {r["id"]: r for r in resources}

    n_trust = int(target_total * 0.25)
    n_sr = int(target_total * 0.40)
    n_allied = int(target_total * 0.20)
    n_rr = int(target_total * 0.15)

    # Provider-Provider TRUST relations
    pairs_used = set()
    while len([r for r in relations if r["type"] == "TRUST"]) < n_trust:
        pi = random.choice(providers)
        pj = random.choice(providers)
        if pi["id"] == pj["id"] or (pi["id"], pj["id"]) in pairs_used:
            continue
        pairs_used.add((pi["id"], pj["id"]))
        relations.append({"head": pi["id"], "relation": "TRUST", "tail": pj["id"],
                           "type": "TRUST", "head_type": "Provider", "tail_type": "Provider"})

    # Service-Resource SUPPORT/OPPOSE/NEUTRAL
    pairs_used = set()
    while len([r for r in relations if r["head_type"] == "Service"]) < n_sr:
        s = random.choice(services)
        r = random.choice(resources)
        if (s["id"], r["id"]) in pairs_used:
            continue
        pairs_used.add((s["id"], r["id"]))
        compat = is_compatible(s, r)
        rand = random.random()
        if rand < conflict_density:
            rel = "OPPOSE"
        elif compat:
            rel = "SUPPORT"
        else:
            rel = "NEUTRAL"
        relations.append({"head": s["id"], "relation": rel, "tail": r["id"],
                           "type": rel, "head_type": "Service", "tail_type": "Resource"})

    # Service-Service ALLIED/CONFLICT relations
    pairs_used = set()
    while len([r for r in relations if r["head_type"] == "Service" and r["tail_type"] == "Service"]) < n_allied:
        si = random.choice(services)
        sj = random.choice(services)
        if si["id"] == sj["id"] or (si["id"], sj["id"]) in pairs_used:
            continue
        pairs_used.add((si["id"], sj["id"]))
        compat = is_compatible(si, sj)
        rand = random.random()
        if rand < conflict_density:
            rel = "CONFLICT"
        else:
            rel = "ALLIED"
        relations.append({"head": si["id"], "relation": rel, "tail": sj["id"],
                           "type": rel, "head_type": "Service", "tail_type": "Service"})

    # Resource-Resource CONFLICT/TRUST
    pairs_used = set()
    while len([r for r in relations if r["head_type"] == "Resource"]) < n_rr:
        ri = random.choice(resources)
        rj = random.choice(resources)
        if ri["id"] == rj["id"] or (ri["id"], rj["id"]) in pairs_used:
            continue
        pairs_used.add((ri["id"], rj["id"]))
        compat = is_compatible(ri, rj)
        rand = random.random()
        if rand < conflict_density:
            rel = "CONFLICT"
        else:
            rel = "TRUST"
        relations.append({"head": ri["id"], "relation": rel, "tail": rj["id"],
                           "type": rel, "head_type": "Resource", "tail_type": "Resource"})

    return relations


def generate_dataset(num_providers: int, num_services: int, num_resources: int,
                      num_relations: int, conflict_density: float = 0.2,
                      seed: int = 42) -> dict:
    """Build the full ICPS hybrid dataset."""
    random.seed(seed)
    logger.info("Generating providers ...")
    providers = generate_providers(num_providers)
    logger.info("Generating services ...")
    services = generate_services(num_services, providers)
    logger.info("Generating resources ...")
    resources = generate_resources(num_resources, providers)
    logger.info("Generating trust/conflict relations ...")
    relations = generate_relations(providers, services, resources, num_relations, conflict_density)

    dataset = {
        "metadata": {
            "num_providers": len(providers),
            "num_services": len(services),
            "num_resources": len(resources),
            "num_relations": len(relations),
            "conflict_density": conflict_density,
            "seed": seed,
            "author": "H. Mezni"
        },
        "providers": providers,
        "services": services,
        "resources": resources,
        "relations": relations
    }
    # Relation type summary
    from collections import Counter
    rcnt = Counter(r["type"] for r in relations)
    dataset["metadata"]["relation_counts"] = dict(rcnt)
    logger.info(f"Dataset generated: {len(providers)} providers, {len(services)} services, "
                f"{len(resources)} resources, {len(relations)} relations.")
    logger.info(f"Relation distribution: {dict(rcnt)}")
    return dataset


def main():
    parser = argparse.ArgumentParser(description="Generate the hybrid ICPS dataset")
    parser.add_argument("--providers", type=int, default=500)
    parser.add_argument("--services", type=int, default=2000)
    parser.add_argument("--resources", type=int, default=1000)
    parser.add_argument("--relations", type=int, default=5000)
    parser.add_argument("--conflict_density", type=float, default=0.2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--out", type=str, default="data/raw/dataset.json")
    args = parser.parse_args()

    dataset = generate_dataset(
        args.providers, args.services, args.resources,
        args.relations, args.conflict_density, args.seed
    )
    os.makedirs(os.path.dirname(args.out), exist_ok=True)
    with open(args.out, "w") as f:
        json.dump(dataset, f, indent=2)
    logger.info(f"Dataset saved to {args.out}")


if __name__ == "__main__":
    main()


Writing data/gen_dataset.py


## 13. Experiment instance splitter (`data/split_dataset.py`)
*Fix applied: the workflow-complexity instance (`tkg_workflow.json`) is now generated with an explicit `sample_instance(...)` call instead of silently reusing the `inst` variable left over from the trust-threshold instance loop.*

In [13]:
%%writefile data/split_dataset.py
"""
split_dataset.py - Generate dataset instances for different experimental configurations.
Author: H. Mezni
"""

import json
import os
import random
import logging
import argparse
import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)


def load_dataset(path: str) -> dict:
    with open(path) as f:
        return json.load(f)


def sample_instance(dataset: dict, num_nodes: int, conflict_density: float, seed: int = 42) -> dict:
    """
    Create a TKG instance with given node count and conflict density.
    num_nodes = total providers + services + resources.
    """
    random.seed(seed)
    providers = dataset["providers"]
    services = dataset["services"]
    resources = dataset["resources"]
    relations = dataset["relations"]

    # Proportional sampling
    ratio_p = len(providers) / (len(providers) + len(services) + len(resources))
    ratio_s = len(services) / (len(providers) + len(services) + len(resources))
    ratio_r = len(resources) / (len(providers) + len(services) + len(resources))

    n_p = max(1, int(num_nodes * ratio_p))
    n_s = max(1, int(num_nodes * ratio_s))
    n_r = max(1, int(num_nodes * ratio_r))
    n_p = min(n_p, len(providers))
    n_s = min(n_s, len(services))
    n_r = min(n_r, len(resources))

    sampled_providers = random.sample(providers, n_p)
    sampled_services = random.sample(services, n_s)
    sampled_resources = random.sample(resources, n_r)

    p_ids = {p["id"] for p in sampled_providers}
    s_ids = {s["id"] for s in sampled_services}
    r_ids = {r["id"] for r in sampled_resources}
    all_ids = p_ids | s_ids | r_ids

    # Filter relations to sampled entities
    valid_rels = [r for r in relations if r["head"] in all_ids and r["tail"] in all_ids]

    # Adjust conflict density: re-tag oppose/conflict edges
    non_conflict_types = {"TRUST", "SUPPORT", "ALLIED"}
    conflict_types = {"OPPOSE", "CONFLICT"}
    adjusted = []
    for rel in valid_rels:
        r = dict(rel)
        if rel["type"] in non_conflict_types:
            if random.random() < conflict_density:
                # Flip to conflict
                if rel["head_type"] == "Service" and rel["tail_type"] == "Resource":
                    r["relation"] = r["type"] = "OPPOSE"
                elif rel["head_type"] == "Service" and rel["tail_type"] == "Service":
                    r["relation"] = r["type"] = "CONFLICT"
                elif rel["head_type"] == "Resource" and rel["tail_type"] == "Resource":
                    r["relation"] = r["type"] = "CONFLICT"
        adjusted.append(r)

    from collections import Counter
    rcnt = Counter(r["type"] for r in adjusted)

    instance = {
        "metadata": {
            "num_nodes": n_p + n_s + n_r,
            "num_providers": n_p,
            "num_services": n_s,
            "num_resources": n_r,
            "num_relations": len(adjusted),
            "conflict_density": conflict_density,
            "seed": seed,
            "relation_counts": dict(rcnt)
        },
        "providers": sampled_providers,
        "services": sampled_services,
        "resources": sampled_resources,
        "relations": adjusted
    }
    return instance


def main():
    parser = argparse.ArgumentParser(description="Create dataset instances for experiments")
    parser.add_argument("--dataset", type=str, default="data/raw/dataset.json")
    parser.add_argument("--out_dir", type=str, default="data/instances")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    dataset = load_dataset(args.dataset)
    os.makedirs(args.out_dir, exist_ok=True)

    # Instances for ICPS size experiment: 1000, 1500, 2000, 2500, 3000 nodes
    for size in [1000, 1500, 2000, 2500, 3000]:
        inst = sample_instance(dataset, size, conflict_density=0.2, seed=args.seed)
        path = os.path.join(args.out_dir, f"tkg_size_{size}.json")
        with open(path, "w") as f:
            json.dump(inst, f, indent=2)
        logger.info(f"Saved instance size={size} -> {path}")

    # Instances for conflict density experiment: fixed 2000 nodes, density 10%-50%
    for density in [0.1, 0.2, 0.3, 0.4, 0.5]:
        inst = sample_instance(dataset, 2000, conflict_density=density, seed=args.seed)
        path = os.path.join(args.out_dir, f"tkg_density_{int(density*100)}.json")
        with open(path, "w") as f:
            json.dump(inst, f, indent=2)
        logger.info(f"Saved instance density={density} -> {path}")

    # Instance for trust threshold experiment: 2000 nodes, 20% conflict
    inst = sample_instance(dataset, 2000, conflict_density=0.2, seed=args.seed)
    path = os.path.join(args.out_dir, "tkg_threshold.json")
    with open(path, "w") as f:
        json.dump(inst, f, indent=2)
    logger.info(f"Saved threshold instance -> {path}")

    # Instance for workflow complexity: 2000 nodes, 20% conflict
    inst = sample_instance(dataset, 2000, conflict_density=0.2, seed=args.seed)
    path = os.path.join(args.out_dir, "tkg_workflow.json")
    with open(path, "w") as f:
        json.dump(inst, f, indent=2)
    logger.info(f"Saved workflow complexity instance -> {path}")

    # First-test small instance: 200 nodes
    inst_small = sample_instance(dataset, 200, conflict_density=0.2, seed=args.seed)
    path = os.path.join(args.out_dir, "tkg_test.json")
    with open(path, "w") as f:
        json.dump(inst_small, f, indent=2)
    logger.info(f"Saved test instance -> {path}")

    logger.info("All dataset instances created.")


if __name__ == "__main__":
    main()


Writing data/split_dataset.py


## 14. Entry-point scripts

In [14]:
%%writefile main-tkg.py
"""
main-tkg.py - Main file for Trust Knowledge Graph construction.
Loads dataset, builds TKG, exports graph structure.
Author: H. Mezni
"""

import os
import sys
import json
import argparse
import logging

# Allow imports from project root
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from src.utils.utils import setup_logging, load_json, save_json, load_config
from src.tkg.tkg import TrustKnowledgeGraph

logger = logging.getLogger(__name__)


def main():
    parser = argparse.ArgumentParser(description="Build Trust Knowledge Graph from dataset")
    parser.add_argument("--config", type=str, default="config.json")
    parser.add_argument("--dataset", type=str, default=None, help="Override dataset path")
    parser.add_argument("--out", type=str, default=None, help="Override TKG output path")
    args = parser.parse_args()

    cfg = load_config(args.config)
    log_dir = cfg["paths"].get("logs", "logs")
    setup_logging(log_dir)

    dataset_path = args.dataset or cfg["paths"]["dataset"]
    tkg_out = args.out or cfg["paths"]["tkg_output"]

    logger.info("=== Trust Knowledge Graph Construction ===")
    logger.info(f"Dataset: {dataset_path}")

    if not os.path.exists(dataset_path):
        logger.error(f"Dataset not found: {dataset_path}. Run data/gen_dataset.py first.")
        sys.exit(1)

    # Load dataset and build TKG
    dataset = load_json(dataset_path)
    tkg = TrustKnowledgeGraph()
    tkg.load_from_dataset(dataset)

    # Print TKG statistics
    stats = tkg.stats()
    logger.info("TKG Statistics:")
    for k, v in stats.items():
        logger.info(f"  {k}: {v}")

    # Export TKG to JSON
    tkg.to_json(tkg_out)
    logger.info(f"TKG structure saved to: {tkg_out}")


if __name__ == "__main__":
    main()


Writing main-tkg.py


In [15]:
%%writefile main-exp.py
"""
main-exp.py - Generate all dataset instances needed for experiments.
Author: H. Mezni
"""

import os
import sys
import argparse
import logging
import subprocess

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from src.utils.utils import setup_logging, load_config

logger = logging.getLogger(__name__)


def run(cmd: list):
    logger.info(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=False, text=True)
    if result.returncode != 0:
        logger.error(f"Command failed: {' '.join(cmd)}")
    return result.returncode


def main():
    parser = argparse.ArgumentParser(description="Generate experiment datasets and instances")
    parser.add_argument("--config", type=str, default="config.json")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    cfg = load_config(args.config)
    setup_logging(cfg["paths"].get("logs", "logs"))

    logger.info("=== Generating main dataset ===")
    ds_cfg = cfg["tkg"]
    run([sys.executable, "data/gen_dataset.py",
         "--providers", str(ds_cfg["num_providers"]),
         "--services", str(ds_cfg["num_services"]),
         "--resources", str(ds_cfg["num_resources"]),
         "--relations", str(ds_cfg["num_relations"]),
         "--conflict_density", "0.2",
         "--seed", str(args.seed),
         "--out", cfg["paths"]["dataset"]])

    logger.info("=== Splitting dataset into experiment instances ===")
    run([sys.executable, "data/split_dataset.py",
         "--dataset", cfg["paths"]["dataset"],
         "--out_dir", "data/instances",
         "--seed", str(args.seed)])

    logger.info("=== All experiment instances ready ===")


if __name__ == "__main__":
    main()


Writing main-exp.py


In [16]:
%%writefile trust-mpgnn.py
"""
trust-mpgnn.py - Main prototype runner for Trust-MPGNN.
Runs the full pipeline: TKG -> Sampling -> Embedding -> Prediction -> Composition.
Author: H. Mezni
"""

import os
import sys
import json
import time
import argparse
import logging
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from src.utils.utils import setup_logging, load_json, save_json, load_config
from src.tkg.tkg import TrustKnowledgeGraph
from src.tkg.metapaths import METAPATHS
from src.gnn.sampler import MetapathSampler
from src.gnn.model import TrustMPGNN
from src.gnn.trainer import TrustTrainer
from src.composition.composer import IoTComposer
from src.composition.workflow import generate_workflow, SAMPLE_QUERIES

logger = logging.getLogger(__name__)


def run_pipeline(cfg: dict, dataset_path: str, workflow_size: int = 10,
                 trust_threshold: float = None):
    """
    Full Trust-MPGNN pipeline:
    1. TKG construction
    2. Metapath-guided sampling
    3. Trust embedding (GNN training)
    4. Trust/conflict prediction
    5. IoT service composition
    """
    t0 = time.time()
    gnn_cfg = cfg["gnn"]
    if trust_threshold is not None:
        gnn_cfg["trust_threshold"] = trust_threshold

    # -- Step 1: TKG Construction --
    logger.info("--- Step 1: TKG Construction ---")
    dataset = load_json(dataset_path)
    tkg = TrustKnowledgeGraph()
    tkg.load_from_dataset(dataset)
    tkg.to_json(cfg["paths"]["tkg_output"])
    stats = tkg.stats()
    logger.info(f"TKG: {stats}")

    # -- Step 2: Metapath-guided Sampling --
    logger.info("--- Step 2: Metapath-guided Sampling ---")
    sampler = MetapathSampler(tkg, METAPATHS, sampling_size=gnn_cfg["sampling_size"])
    neighborhoods = sampler.build_neighborhoods()
    nbr_idx_stack, nbr_mask_stack = sampler.to_padded_tensors(neighborhoods)

    # -- Step 3: Trust Embedding (Algorithm 1) --
    logger.info("--- Step 3: Trust Embedding ---")
    model = TrustMPGNN(
        input_dim=16,
        embed_dim=gnn_cfg["embed_dim"],
        num_layers=gnn_cfg["num_layers"],
        num_metapaths=len(METAPATHS),
        dropout=gnn_cfg["dropout"]
    )
    trainer = TrustTrainer(model, tkg, sampler, gnn_cfg)
    H = trainer.train(nbr_idx_stack, nbr_mask_stack)
    trainer.save_embeddings(H, cfg["paths"]["embeddings"])

    # -- Step 4: Trust Prediction --
    logger.info("--- Step 4: Trust/Conflict Prediction ---")
    E_hat, Delta, Gamma = trainer.predict_relations(H)

    # -- Step 5: Composition (Algorithm 2) --
    logger.info(f"--- Step 5: IoT Service Composition (workflow={workflow_size}) ---")
    workflow = generate_workflow(workflow_size, seed=42)
    composer = IoTComposer(tkg, H, Delta, Gamma, cfg["composition"])
    result = composer.compose(workflow)
    result["trust_score"] = composer.trust_score(result["assignments"])
    result["conflict_severity"] = composer.conflict_severity(result["assignments"])
    result["elapsed_sec"] = round(time.time() - t0, 2)

    logger.info(f"Composition complete: score={result['composition_score']}, "
                f"trust={result['trust_score']}, severity={result['conflict_severity']}")

    return result, E_hat, Delta, Gamma, H, tkg


def main():
    parser = argparse.ArgumentParser(description="Run Trust-MPGNN prototype")
    parser.add_argument("--config", type=str, default="config.json")
    parser.add_argument("--dataset", type=str, default=None)
    parser.add_argument("--workflow", type=int, default=10, help="Workflow size")
    parser.add_argument("--theta", type=float, default=None, help="Trust threshold")
    parser.add_argument("--query", type=int, default=None, help="Use predefined query (1-4)")
    parser.add_argument("--out", type=str, default="output/composition_result.json")
    args = parser.parse_args()

    cfg = load_config(args.config)
    setup_logging(cfg["paths"].get("logs", "logs"))

    dataset_path = args.dataset or cfg["paths"]["dataset"]
    if not os.path.exists(dataset_path):
        logger.error(f"Dataset not found: {dataset_path}. Run: python main-exp.py")
        sys.exit(1)

    logger.info("=== Trust-MPGNN: Conflict-aware IoT Service Composition ===")
    logger.info(f"Author: {cfg.get('author', 'H. Mezni')}")

    # Choose workflow
    if args.query and 1 <= args.query <= len(SAMPLE_QUERIES):
        q = SAMPLE_QUERIES[args.query - 1]
        logger.info(f"Using predefined query Q{args.query}: {q['description']}")
        workflow_size = len(q["workflow"])
    else:
        workflow_size = args.workflow

    result, E_hat, Delta, Gamma, H, tkg = run_pipeline(
        cfg, dataset_path, workflow_size, args.theta
    )

    # Output
    output = {
        "pipeline": "Trust-MPGNN",
        "author": "H. Mezni",
        "dataset": dataset_path,
        "workflow_size": workflow_size,
        "trust_threshold": cfg["gnn"]["trust_threshold"],
        "tkg_stats": tkg.stats(),
        "trust_relations": len(Delta),
        "conflict_relations": len(Gamma),
        "composition": result
    }

    os.makedirs(os.path.dirname(args.out) if os.path.dirname(args.out) else ".", exist_ok=True)
    save_json(output, args.out)
    logger.info(f"\n{'='*50}")
    logger.info("COMPOSITION RESULT:")
    logger.info(f"  Trust Score:       {result['trust_score']}")
    logger.info(f"  Conflict Severity: {result['conflict_severity']}")
    logger.info(f"  Composition Score: {result['composition_score']}")
    logger.info(f"  Services Used:     {result['num_services']}")
    logger.info(f"  Resources Used:    {result['num_resources']}")
    logger.info(f"  Elapsed:           {result['elapsed_sec']}s")
    logger.info(f"  Results saved:     {args.out}")
    logger.info(f"{'='*50}")

    # Print assignments
    logger.info("\nTask Assignments:")
    for a in result["assignments"]:
        logger.info(f"  [{a['task_id']}] {a.get('task_name','')} -> "
                    f"Service: {a['service_name']} | Resource: {a['resource_name']} | "
                    f"Trust: {a['proximity_degree']}")


if __name__ == "__main__":
    main()


Writing trust-mpgnn.py


In [17]:
%%writefile tests/test_prototype.py
"""
test_prototype.py - First test: validate Trust-MPGNN prototype on small TKG + sample queries.
Author: H. Mezni
"""

import os
import sys
import logging
import json

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.utils.utils import setup_logging, load_json, save_json, load_config
from src.tkg.tkg import TrustKnowledgeGraph
from src.tkg.metapaths import METAPATHS
from src.gnn.sampler import MetapathSampler
from src.gnn.model import TrustMPGNN
from src.gnn.trainer import TrustTrainer
from src.composition.composer import IoTComposer
from src.composition.workflow import SAMPLE_QUERIES
from src.composition.baselines import compute_metrics

logger = logging.getLogger(__name__)


def main():
    cfg = load_config(os.path.join(os.path.dirname(os.path.dirname(__file__)), "config.json"))
    # Use fewer epochs for quick test
    cfg["gnn"]["epochs"] = 30
    cfg["gnn"]["sampling_size"] = 10
    setup_logging(cfg["paths"].get("logs", "logs"))

    logger.info("=== First Test: Trust-MPGNN Prototype Validation ===")
    logger.info("Author: H. Mezni")

    inst_path = "data/instances/tkg_test.json"
    if not os.path.exists(inst_path):
        logger.error("Test instance not found. Run: python main-exp.py")
        sys.exit(1)

    dataset = load_json(inst_path)
    logger.info(f"Test TKG: {dataset['metadata']['num_nodes']} nodes, "
                f"{dataset['metadata']['num_relations']} relations")

    # Build TKG
    tkg = TrustKnowledgeGraph()
    tkg.load_from_dataset(dataset)
    stats = tkg.stats()
    logger.info(f"TKG stats: {stats}")

    # Sample + train
    sampler = MetapathSampler(tkg, METAPATHS, sampling_size=cfg["gnn"]["sampling_size"])
    neighborhoods = sampler.build_neighborhoods()
    nbr_idx_stack, nbr_mask_stack = sampler.to_padded_tensors(neighborhoods)

    model = TrustMPGNN(16, cfg["gnn"]["embed_dim"], 2, len(METAPATHS), cfg["gnn"]["dropout"])
    trainer = TrustTrainer(model, tkg, sampler, cfg["gnn"])
    H = trainer.train(nbr_idx_stack, nbr_mask_stack)
    _, Delta, Gamma = trainer.predict_relations(H)
    conflict_set = {(u, v) for u, s, v in Gamma}

    logger.info(f"\nTrust relations (Delta): {len(Delta)}")
    logger.info(f"Conflict relations (Gamma): {len(Gamma)}")

    # Test all sample queries
    all_results = []
    for q in SAMPLE_QUERIES:
        logger.info(f"\n--- Query {q['query_id']}: {q['description']} ---")
        workflow = q["workflow"]
        composer = IoTComposer(tkg, H, Delta, Gamma, cfg["composition"])
        result = composer.compose(workflow)
        metrics = compute_metrics(result, conflict_set, tkg)

        logger.info(f"  Workflow size: {len(workflow)}")
        logger.info(f"  Trust score:       {metrics['trust_score']:.3f}")
        logger.info(f"  Conflict severity: {metrics['conflict_severity']:.3f}")
        logger.info(f"  Success rate:      {metrics['success_rate']:.3f}")
        logger.info(f"  Composition score: {result['composition_score']:.4f}")
        logger.info("  Assignments:")
        for a in result["assignments"]:
            logger.info(f"    [{a['task_id']}] {a.get('task_name','')} -> "
                        f"Service: {a['service_name'][:30]:30s} | "
                        f"Resource: {a['resource_name'][:30]:30s} | Trust={a['proximity_degree']:.3f}")

        all_results.append({
            "query_id": q["query_id"],
            "description": q["description"],
            "metrics": metrics,
            "composition_score": result["composition_score"],
            "assignments": result["assignments"]
        })

    out_path = "output/test_results.json"
    save_json({"test": "prototype_validation", "results": all_results}, out_path)
    logger.info(f"\nAll test results saved to {out_path}")
    logger.info("=== Test Complete ===")


if __name__ == "__main__":
    main()


Writing tests/test_prototype.py


## 15. Run it — generate data, build the TKG, train, predict, compose

This uses a **reduced-size** ICPS for a fast Colab run (still produces meaningful results). Bump `NUM_PROVIDERS / NUM_SERVICES / NUM_RESOURCES / NUM_RELATIONS` and `EPOCHS` back up to the paper's defaults (500 / 2000 / 1000 / 5000, 200 epochs) below if you have time / a GPU runtime and want to reproduce the paper's exact scale — with the vectorized GNN this notebook now also runs the *full* default config (2000-node instance, 200 epochs) in a few minutes on CPU.

In [18]:
import json

with open("config.json") as f:
    cfg = json.load(f)

# Reduced sizes for a fast end-to-end demo run. Set RUN_FULL_SCALE = True to
# use the paper's original defaults from config.json instead.
RUN_FULL_SCALE = False

if not RUN_FULL_SCALE:
    cfg["tkg"]["num_providers"] = 80
    cfg["tkg"]["num_services"] = 300
    cfg["tkg"]["num_resources"] = 150
    cfg["tkg"]["num_relations"] = 800
    cfg["gnn"]["epochs"] = 80
    with open("config.json", "w") as f:
        json.dump(cfg, f, indent=2)

print(json.dumps(cfg["tkg"], indent=2))
print(json.dumps(cfg["gnn"], indent=2))

{
  "num_providers": 80,
  "num_services": 300,
  "num_resources": 150,
  "num_relations": 800,
  "relation_types": [
    "TRUST",
    "SUPPORT",
    "OPPOSE",
    "NEUTRAL",
    "ALLIED",
    "CONFLICT"
  ]
}
{
  "embed_dim": 128,
  "num_layers": 2,
  "num_heads": 4,
  "dropout": 0.3,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "batch_size": 256,
  "epochs": 80,
  "trust_threshold": 0.6,
  "sampling_size": 20
}


In [19]:
# Step A: generate the ICPS dataset + all per-experiment instances
!python main-exp.py --seed 42

2026-06-22 23:21:43,531 [INFO] __main__: === Generating main dataset ===
2026-06-22 23:21:43,531 [INFO] __main__: Running: /usr/bin/python3 data/gen_dataset.py --providers 80 --services 300 --resources 150 --relations 800 --conflict_density 0.2 --seed 42 --out data/raw/dataset.json
2026-06-22 23:21:43,618 [INFO] Generating providers ...
2026-06-22 23:21:43,619 [INFO] Generating services ...
2026-06-22 23:21:43,633 [INFO] Generating resources ...
2026-06-22 23:21:43,639 [INFO] Generating trust/conflict relations ...
2026-06-22 23:21:43,660 [INFO] Dataset generated: 80 providers, 300 services, 150 resources, 800 relations.
2026-06-22 23:21:43,660 [INFO] Relation distribution: {'TRUST': 290, 'SUPPORT': 167, 'NEUTRAL': 92, 'OPPOSE': 61, 'ALLIED': 129, 'CONFLICT': 61}
2026-06-22 23:21:43,687 [INFO] Dataset saved to data/raw/dataset.json
2026-06-22 23:21:43,697 [INFO] __main__: === Splitting dataset into experiment instances ===
2026-06-22 23:21:43,697 [INFO] __main__: Running: /usr/bin/pyth

In [20]:
# Step B: build the Trust Knowledge Graph and export its structure
!python main-tkg.py

2026-06-22 23:21:45,659 [INFO] __main__: === Trust Knowledge Graph Construction ===
2026-06-22 23:21:45,659 [INFO] __main__: Dataset: data/raw/dataset.json
2026-06-22 23:21:45,672 [INFO] src.tkg.tkg: Building Trust Knowledge Graph ...
2026-06-22 23:21:45,680 [INFO] src.tkg.tkg: TKG built: 530 nodes, 800 edges.
2026-06-22 23:21:45,681 [INFO] __main__: TKG Statistics:
2026-06-22 23:21:45,681 [INFO] __main__:   num_nodes: 530
2026-06-22 23:21:45,682 [INFO] __main__:   num_edges: 800
2026-06-22 23:21:45,682 [INFO] __main__:   node_types: {'Provider': 80, 'Service': 300, 'Resource': 150}
2026-06-22 23:21:45,682 [INFO] __main__:   edge_types: {'TRUST': 290, 'SUPPORT': 167, 'NEUTRAL': 92, 'CONFLICT': 61, 'ALLIED': 129, 'OPPOSE': 61}
2026-06-22 23:21:45,715 [INFO] src.tkg.tkg: TKG exported to output/tkg.json
2026-06-22 23:21:45,715 [INFO] __main__: TKG structure saved to: output/tkg.json


In [21]:
# Step C: run the full Trust-MPGNN prototype (TKG -> sampling -> embedding -> prediction -> composition)
!python trust-mpgnn.py --workflow 10

2026-06-22 23:21:51,760 [INFO] __main__: === Trust-MPGNN: Conflict-aware IoT Service Composition ===
2026-06-22 23:21:51,760 [INFO] __main__: Author: H. Mezni
2026-06-22 23:21:51,760 [INFO] __main__: --- Step 1: TKG Construction ---
2026-06-22 23:21:51,774 [INFO] src.tkg.tkg: Building Trust Knowledge Graph ...
2026-06-22 23:21:51,799 [INFO] src.tkg.tkg: TKG built: 530 nodes, 800 edges.
2026-06-22 23:21:51,843 [INFO] src.tkg.tkg: TKG exported to output/tkg.json
2026-06-22 23:21:51,845 [INFO] __main__: TKG: {'num_nodes': 530, 'num_edges': 800, 'node_types': {'Provider': 80, 'Service': 300, 'Resource': 150}, 'edge_types': {'TRUST': 290, 'SUPPORT': 167, 'NEUTRAL': 92, 'CONFLICT': 61, 'ALLIED': 129, 'OPPOSE': 61}}
2026-06-22 23:21:51,845 [INFO] __main__: --- Step 2: Metapath-guided Sampling ---
2026-06-22 23:21:51,845 [INFO] src.gnn.sampler: Building metapath neighborhoods for 530 nodes ...
2026-06-22 23:21:51,924 [INFO] src.gnn.sampler:   Sampled 500/530 nodes ...
2026-06-22 23:21:51,927 [

In [22]:
import json
with open("output/composition_result.json") as f:
    result = json.load(f)

print(f"Trust threshold (theta):  {result['trust_threshold']}")
print(f"Trust relations:          {result['trust_relations']}")
print(f"Conflict relations:       {result['conflict_relations']}")
print(f"Trust score:              {result['composition']['trust_score']}")
print(f"Conflict severity:        {result['composition']['conflict_severity']}")
print(f"Composition score:        {result['composition']['composition_score']}")
print()
for a in result["composition"]["assignments"]:
    print(f"[{a['task_id']}] {a.get('task_name',''):20s} -> "
          f"Service: {a['service_name']:20s} | Resource: {a['resource_name']:20s} | "
          f"Trust={a['proximity_degree']:.3f}")

Trust threshold (theta):  0.6
Trust relations:          436
Conflict relations:       364
Trust score:              0.9458
Conflict severity:        0.0542
Composition score:        1.5074

[T01] EnergyManage_1       -> Service: FitnessTracker_131   | Resource: BioSensor_106        | Trust=1.000
[T02] HealthMonitor_2      -> Service: InventoryManager_211 | Resource: FogNode_109          | Trust=1.000
[T03] TrafficMonitor_3     -> Service: IrrigationControl_26 | Resource: SmartMeter_Res_108   | Trust=1.000
[T04] AirQuality_4         -> Service: AccessControl_120    | Resource: SmartLock_Res_113    | Trust=1.000
[T05] AirQuality_5         -> Service: AttendanceTracker_130 | Resource: DoorSensor_61        | Trust=1.000
[T06] SecurityMonitor_6    -> Service: ClimateControl_245   | Resource: WearableDevice_59    | Trust=1.000
[T07] EnergyManage_7       -> Service: SmartMeter_90        | Resource: BatteryPack_120      | Trust=1.000
[T08] AccessControl_8      -> Service: AttendanceTracker_234

## 16. First-test validation — 4 predefined IoT queries on a small TKG

In [23]:
!python tests/test_prototype.py

2026-06-22 23:22:43,551 [INFO] __main__: === First Test: Trust-MPGNN Prototype Validation ===
2026-06-22 23:22:43,551 [INFO] __main__: Author: H. Mezni
2026-06-22 23:22:43,555 [INFO] __main__: Test TKG: 199 nodes, 114 relations
2026-06-22 23:22:43,555 [INFO] src.tkg.tkg: Building Trust Knowledge Graph ...
2026-06-22 23:22:43,557 [INFO] src.tkg.tkg: TKG built: 199 nodes, 114 edges.
2026-06-22 23:22:43,557 [INFO] __main__: TKG stats: {'num_nodes': 199, 'num_edges': 114, 'node_types': {'Provider': 30, 'Service': 113, 'Resource': 56}, 'edge_types': {'TRUST': 42, 'ALLIED': 16, 'NEUTRAL': 14, 'OPPOSE': 16, 'SUPPORT': 16, 'CONFLICT': 10}}
2026-06-22 23:22:43,557 [INFO] src.gnn.sampler: Building metapath neighborhoods for 199 nodes ...
2026-06-22 23:22:43,565 [INFO] src.gnn.sampler: Neighborhood sampling complete.
2026-06-22 23:22:44,790 [INFO] src.gnn.trainer: Training TrustMPGNN: 199 nodes, 80 edge samples, 30 epochs ...
2026-06-22 23:22:44,931 [INFO] src.gnn.trainer:   Epoch 1/30 | Loss: 0.

In [24]:
import json
with open("output/test_results.json") as f:
    test_results = json.load(f)

for r in test_results["results"]:
    m = r["metrics"]
    print(f"{r['query_id']}: {r['description']}")
    print(f"  success_rate={m['success_rate']:.3f}  trust_score={m['trust_score']:.3f}  "
          f"conflict_severity={m['conflict_severity']:.3f}  composition_score={r['composition_score']:.4f}")
    print()

Q1: Smart healthcare monitoring application
  success_rate=1.000  trust_score=0.917  conflict_severity=0.083  composition_score=3.1735

Q2: Smart building energy management
  success_rate=1.000  trust_score=1.000  conflict_severity=0.000  composition_score=22.5101

Q3: Smart mobility and parking
  success_rate=1.000  trust_score=0.936  conflict_severity=0.064  composition_score=4.2575

Q4: Smart home automation
  success_rate=1.000  trust_score=0.984  conflict_severity=0.016  composition_score=10.0091



## For experiments

The experiment scripts in `experiments/` (e.g. impact of conflict density, workflow complexity, trust-threshold sensitivity, ICPS-size scalability, composition time) are more compute-heavy (they train Trust-MPGNN multiple times), and use the *full-scale* dataset instances written to `data/instances/` by `main-exp.py`. Uncomment and run the cell below if you'd like to reproduce one of them (the conflict-density experiment is the lightest one to start with).


In [25]:
# %%writefile experiments/exp_conflict_density.py
# (see the project's experiments/ folder structure — each script loads a
# data/instances/tkg_*.json file, trains Trust-MPGNN once, runs the
# baselines, and writes output/exp_*.json). To keep this notebook's default
# run fast we don't execute an experiment automatically; uncomment the line
# below to run the conflict-density experiment (Table 8) once you've set
# RUN_FULL_SCALE = True above and re-run Steps A/B/C with the full-size data.

# !python experiments/exp_conflict_density.py